Causal test, stage 2: real answers. On the cluster, 8B generated answers (256 tokens) under each push and Qwen3Guard judged them. Pushes: `topup_h` and `fixed_h` add harm along English `h` on the instruction tokens at the push layer (layer 10 on 8B, the cleanest layer in the notebook 10 sweep), `topup_rand` is the random control, `add_r_x0.25` / `add_r_x0.5` add a quarter / half of the refusal vector `r` at `r_layer` on every token (steer-r). We only read label fields (class), never the answers.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml
from harm_route.judge import is_garbled, to_class

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
name = "qwen3-8b"
judge = "qwen3guard-4b"
langs = ["eng_Latn", "npi_Deva", "srp_Cyrl", "srp_Latn"]
cfg = yaml.safe_load(open(ROOT / "configs/experiments/e01_causal_gen.yaml"))
conds = cfg["conditions"]
script = {l: v["script"] for l, v in yaml.safe_load(open(ROOT / "configs/languages.yaml")).items()}
print("conditions", conds)

Read one label file into a list of classes, in prompt order. Same class rule as notebooks 09 and 10.

In [ ]:
def classes(path, lang):
    rows = [json.loads(x) for x in open(path)]
    assert [r["idx"] for r in rows] == list(range(len(rows))), "rows out of order"
    return [to_class(r, is_garbled(r["n_tokens"], r["d4"], r["script"], script[lang])) for r in rows]

C = {l: {st: {c: classes(ROOT / f"data/generations/e01_causal/{name}/{l}/judge/{judge}/{st}__{c}.jsonl", l)
              for c in conds} for st in ("harmful_train", "harmless_val")} for l in langs}
# groups from the ORIGINAL base run (512 tokens, notebook 09): who complied before any push
orig = {l: classes(ROOT / f"data/generations/e01_scale/{name}/{l}/judge/{judge}/harmful_train__base.jsonl", l) for l in langs}
G = {l: torch.tensor([1 if c == "refuse" else 0 if c in ("comply_harmful", "comply_benign") else -1 for c in orig[l]]) for l in langs}

Sanity: does the new base run (256 tokens) agree with the original base run (512 tokens)? Greedy decoding with the same model should give the same start of each answer, so refusals should mostly match. Disagreements come from the shorter answer changing the judge's view (for example, a refusal that leaks later is now cut off).

In [ ]:
for l in langs:
    new = C[l]["harmful_train"]["base"]
    same = sum((a == "refuse") == (b == "refuse") for a, b in zip(orig[l], new))
    print(f"{l:9s} refuse agree {same}/{len(new)} | original refuse {sum(c == 'refuse' for c in orig[l])}, new {sum(c == 'refuse' for c in new)}")

The main table: refusal rate under each push, for the prompts that complied in the original run (the test), the prompts that refused (should stay refused), and harmless prompts (should stay answered; any refusal there is over-refusal). Refusal = a clean refusal (class `refuse`).

In [ ]:
def rate(cls, mask=None):
    keep = [c for i, c in enumerate(cls) if mask is None or mask[i]]
    return sum(c == "refuse" for c in keep) / max(len(keep), 1), len(keep)

T = {}
print(f"{'lang':9s} {'group':9s} {'n':>3s} " + " ".join(f"{c:>11s}" for c in conds))
for l in langs:
    for gname, st, mask in [("complied", "harmful_train", G[l] == 0), ("refused", "harmful_train", G[l] == 1),
                            ("harmless", "harmless_val", None)]:
        vals = [rate(C[l][st][c], mask) for c in conds]
        if vals[0][1] == 0:
            continue
        T[(l, gname)] = [v for v, _ in vals]
        print(f"{l:9s} {gname:9s} {vals[0][1]:3d} " + " ".join(f"{v:11.2f}" for v, _ in vals))

The same as a picture: refusal rate of the original compliers under each push, per language, with harmless over-refusal as a hatched bar next to it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
x = torch.arange(len(conds)).float()
for ax, l in zip(axes, langs[1:]):
    ax.bar(x - 0.2, T[(l, "complied")], width=0.4, label="original compliers: refuse rate")
    ax.bar(x + 0.2, T[(l, "harmless")], width=0.4, hatch="//", fill=False, label="harmless: refuse rate (over-refusal)")
    ax.set_xticks(x, conds, rotation=30, ha="right", fontsize=8); ax.set_title(l); ax.set_ylim(0, 1)
axes[0].set_ylabel("fraction refused"); axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()

What the compliers turn into under each push: the full class breakdown (refuse, refuse_leak, comply_harmful, comply_benign, garbled). A push that 'works' by breaking the output into garbage is not a real fix, so the garbled count matters.

In [ ]:
from collections import Counter
for l in langs[1:]:
    m = G[l] == 0
    print(f"\n{l} original compliers (n={int(m.sum())})")
    for c in conds:
        cnt = Counter(x for i, x in enumerate(C[l]["harmful_train"][c]) if m[i])
        print(f"  {c:12s} " + "  ".join(f"{k}:{v}" for k, v in sorted(cnt.items())))